In [ ]:
import pandas as pd
import os

# 1. 파일 경로 설정 (r을 붙여 백슬래시 경로 인식)
path_overdue = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_OVERDUE.csv"
path_moratorium = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_MORATORIUM.csv"

def audit_file(file_path, file_label):
    print(f"\n{'='*30} [{file_label}] Audit {'='*30}")
    
    if not os.path.exists(file_path):
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return None

    # 데이터 로드 (ID는 문자열로 고정)
    df = pd.read_csv(file_path, dtype={'COMPANY_ID': str})
    
    # 기본 제원
    print(f"📊 데이터 규모: {df.shape[0]:,} 행 × {df.shape[1]} 열")
    print(f"🆔 고유 기업 수(Unique ID): {df['COMPANY_ID'].nunique():,}개")
    
    # 결측치 확인
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\n❓ 결측치 발생 컬럼:")
        print(null_counts[null_counts > 0])
    else:
        print("\n✅ 결측치 없음")

    # 수치형 변수(리스크 지표) 분포 확인
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    if num_cols:
        print(f"\n📈 리스크 지표 요약 ({', '.join(num_cols)}):")
        display(df[num_cols].describe().T[['count', 'mean', 'min', 'max']])
        
        # 0 초과(실제 발생) 케이스 확인
        print("\n🚩 실질적 리스크 발생 건수 (Value > 0):")
        for col in num_cols:
            active_cases = (df[col] > 0).sum()
            print(f" - {col}: {active_cases:,}건")

    print("\n📝 데이터 샘플 (Top 3):")
    display(df.head(3))
    return df

# 실행
df_overdue = audit_file(path_overdue, "COMPANY_OVERDUE")
df_moratorium = audit_file(path_moratorium, "COMPANY_MORATORIUM")

# 2. [Cross-Check] 두 파일 간의 교집합 확인
if df_overdue is not None and df_moratorium is not None:
    common_ids = set(df_overdue['COMPANY_ID']) & set(df_moratorium['COMPANY_ID'])
    print(f"\n{'='*80}")
    print(f"🔗 [병합 전 점검] 두 파일 공통 ID 수: {len(common_ids):,}개")
    print(f"💡 Tip: 공통 ID가 적다면 '누락'을, 너무 많다면 '데이터 중복'을 의심해야 합니다.")


============================== [COMPANY_OVERDUE] Audit ==============================
📊 데이터 규모: 34 행 × 25 열
🆔 고유 기업 수(Unique ID): 11개

❓ 결측치 발생 컬럼:
DEBTOR_NAME             12
RELEASE_DATE            34
DELETION_DATE           34
_AB_CDC_DELETED_AT      34
RELEASE_REASON_CODE     34
DELETION_REASON_CODE    34
dtype: int64

📈 리스크 지표 요약 (_AIRBYTE_GENERATION_ID, ID, _AB_CDC_LSN, BOND_NUMBER, RELEASE_DATE, DELETION_DATE, OVERDUE_AMOUNT, _AB_CDC_DELETED_AT, RELEASE_REASON_CODE, DELETION_REASON_CODE, CREDITOR_BUSINESS_NUMBER, REGISTRATION_REASON_CODE, DEBTOR_CLASSIFICATION_CODE):


,count,mean,min,max
_AIRBYTE_GENERATION_ID,34.0,0.000000e+00,0.000000e+00,0.000000e+00
ID,34.0,7.744118e+01,2.500000e+01,1.420000e+02
_AB_CDC_LSN,34.0,2.763220e+08,9.388639e+07,8.207873e+08
BOND_NUMBER,34.0,1.112779e+05,9.983400e+04,1.294410e+05
RELEASE_DATE,0.0,NaN,NaN,NaN
DELETION_DATE,0.0,NaN,NaN,NaN
OVERDUE_AMOUNT,34.0,1.359638e+08,5.000000e+05,1.075130e+09
_AB_CDC_DELETED_AT,0.0,NaN,NaN,NaN
RELEASE_REASON_CODE,0.0,NaN,NaN,NaN
DELETION_REASON_CODE,0.0,NaN,NaN,NaN



🚩 실질적 리스크 발생 건수 (Value > 0):
 - _AIRBYTE_GENERATION_ID: 0건
 - ID: 34건
 - _AB_CDC_LSN: 34건
 - BOND_NUMBER: 34건
 - RELEASE_DATE: 0건
 - DELETION_DATE: 0건
 - OVERDUE_AMOUNT: 34건
 - _AB_CDC_DELETED_AT: 0건
 - RELEASE_REASON_CODE: 0건
 - DELETION_REASON_CODE: 0건
 - CREDITOR_BUSINESS_NUMBER: 34건
 - REGISTRATION_REASON_CODE: 34건
 - DEBTOR_CLASSIFICATION_CODE: 34건

📝 데이터 샘플 (Top 3):


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,BOND_NUMBER,DEBTOR_NAME,...,REGISTRATION_DATE,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,DEBTOR_COMPANY_NAME,RELEASE_REASON_CODE,DELETION_REASON_CODE,CREDITOR_BUSINESS_NUMBER,REGISTRATION_REASON_CODE,DEBTOR_CLASSIFICATION_CODE,REGISTRATION_REASON_OCCURRENCE_DATE
0,e39a9a50-c610-49e7-862f-b8904098b6e4,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,25,73,2025-01-22 15:13:19.230170,93886392.0,99862,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2021-11-09
1,842830fa-d64c-4207-a192-db80b6667bd0,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,27,73,2025-01-22 15:13:19.233167,93886392.0,99840,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2022-03-13
2,3f708280-b41b-4c05-8e55-19d0352ee4b0,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,29,73,2025-01-22 15:13:19.235450,93886392.0,99834,NaN,...,2022-05-05,NaN,2025-10-14T11:26:32.305225016Z,동성금속기술(주),NaN,NaN,2118141143,1,1,2021-11-09



============================== [COMPANY_MORATORIUM] Audit ==============================
📊 데이터 규모: 1,913 행 × 16 열
🆔 고유 기업 수(Unique ID): 1,423개

❓ 결측치 발생 컬럼:
_AB_CDC_DELETED_AT    1913
dtype: int64

📈 리스크 지표 요약 (_AIRBYTE_GENERATION_ID, ID, _AB_CDC_LSN, WORKOUT_COUNT, MORATORIUM_COUNT, _AB_CDC_DELETED_AT, CARD_ACCOUNT_COUNT, ACCOUNT_SUSPENSION_COUNT, MORATORIUM_OVERDUE_AMOUNT, MORATORIUM_REGISTERED_AMOUNT):


,count,mean,min,max
_AIRBYTE_GENERATION_ID,1913.0,0.000000e+00,0.0,0.000000e+00
ID,1913.0,9.990152e+02,1.0,1.970000e+03
_AB_CDC_LSN,1913.0,9.599813e+08,93886392.0,2.833766e+09
WORKOUT_COUNT,1913.0,7.318348e-03,0.0,1.000000e+00
MORATORIUM_COUNT,1913.0,3.753267e-01,0.0,6.900000e+01
_AB_CDC_DELETED_AT,0.0,NaN,NaN,NaN
CARD_ACCOUNT_COUNT,1913.0,5.006796e+00,0.0,4.300000e+01
ACCOUNT_SUSPENSION_COUNT,1913.0,3.659174e-03,0.0,1.000000e+00
MORATORIUM_OVERDUE_AMOUNT,1913.0,1.659643e+05,0.0,1.005031e+08
MORATORIUM_REGISTERED_AMOUNT,1913.0,2.091538e+05,0.0,1.332691e+08



🚩 실질적 리스크 발생 건수 (Value > 0):
 - _AIRBYTE_GENERATION_ID: 0건
 - ID: 1,913건
 - _AB_CDC_LSN: 1,913건
 - WORKOUT_COUNT: 14건
 - MORATORIUM_COUNT: 107건
 - _AB_CDC_DELETED_AT: 0건
 - CARD_ACCOUNT_COUNT: 1,703건
 - ACCOUNT_SUSPENSION_COUNT: 7건
 - MORATORIUM_OVERDUE_AMOUNT: 76건
 - MORATORIUM_REGISTERED_AMOUNT: 96건

📝 데이터 샘플 (Top 3):


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,COMPANY_ID,CREATED_AT,_AB_CDC_LSN,WORKOUT_COUNT,MORATORIUM_COUNT,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,CARD_ACCOUNT_COUNT,ACCOUNT_SUSPENSION_COUNT,MORATORIUM_OVERDUE_AMOUNT,MORATORIUM_REGISTERED_AMOUNT
0,d3650d81-a7cf-4bcc-83ae-bbb4fd3b9097,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,1,36,2025-01-02 05:02:56.610391,93886392.0,0,0,NaN,2025-10-14T11:26:32.305225016Z,4,0,0,0
1,2d8f5388-9497-4955-9171-65e89f68c9dc,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,2,15,2025-01-02 05:14:14.686321,93886392.0,0,0,NaN,2025-10-14T11:26:32.305225016Z,2,0,0,0
2,b7f95d04-9b98-4928-812d-607298c6df49,2025-10-14 11:26:32.305000+00:00,"{\n ""changes"": [],\n ""sync_id"": 23\n}",0,3,1,2025-01-02 07:59:27.954822,93886392.0,0,0,NaN,2025-10-14T11:26:32.305225016Z,6,0,0,0



🔗 [병합 전 점검] 두 파일 공통 ID 수: 11개
💡 Tip: 공통 ID가 적다면 '누락'을, 너무 많다면 '데이터 중복'을 의심해야 합니다.


In [ ]:
import pandas as pd
import numpy as np

print("🧪 [Target Validation] 데이터 타입 보정 후 Y값 정밀 검증...")

# 1. 데이터 로드 (파일 경로 및 ID 타입 설정)
df_overdue = pd.read_csv(r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_OVERDUE.csv", dtype={'COMPANY_ID': str})
df_mora = pd.read_csv(r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_MORATORIUM.csv", dtype={'COMPANY_ID': str})
df_master = pd.read_csv("Final_Integrated_Credit_Master_v5_Income_Added.csv", dtype={'COMPANY_ID': str})

# 2. Y값 후보군 ID 세트 생성
ov_ids = set(df_overdue['COMPANY_ID'].unique())
mo_ids = set(df_mora[df_mora['MORATORIUM_OVERDUE_AMOUNT'] > 0]['COMPANY_ID'].unique())
all_bad_ids = ov_ids | mo_ids

# 3. 데이터 타입 보정 (문자열 -> 숫자)
# 'coerce'는 숫자로 바꿀 수 없는 값을 NaN으로 처리합니다.
cols_to_check = ['OPERATING_MARGIN', 'DEBT_RATIO', 'CASH_RATIO', 'SALES_REVENUE']
for col in cols_to_check:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 4. 임시 Y라벨 부여 (1: 부실 징후, 0: 정상)
df_master['TMP_Y'] = df_master['COMPANY_ID'].apply(lambda x: 1 if x in all_bad_ids else 0)

# 5. [재무 상관관계 대조] Y=1 그룹 vs Y=0 그룹
# 결측치를 제외하고 수치형 데이터만 평균 계산
comparison = df_master.groupby('TMP_Y')[['OPERATING_MARGIN', 'DEBT_RATIO', 'CASH_RATIO']].mean()

print(f"\n📊 [최종 검증: 재무 지표 대조 (Smell Test)]")
print("-" * 60)
if not comparison.empty:
    display(comparison)
    
    # 해석 가이드
    print("\n💡 CIO 체크포인트:")
    print("1. TMP_Y=1(부실군)의 영업이익률(OPERATING_MARGIN)이 정상군보다 낮은가?")
    print("2. TMP_Y=1(부실군)의 부채비율(DEBT_RATIO)이 정상군보다 현저히 높은가?")
    print("👉 두 질문에 'Yes'라면, 우리가 잡은 Y값은 '진짜 리스크'를 대변하는 정답지가 맞습니다.")
else:
    print("❌ 데이터 비교에 실패했습니다. 컬럼명을 다시 확인해주세요.")

🧪 [Target Validation] 데이터 타입 보정 후 Y값 정밀 검증...

📊 [최종 검증: 재무 지표 대조 (Smell Test)]
------------------------------------------------------------


,OPERATING_MARGIN,DEBT_RATIO,CASH_RATIO
TMP_Y,,,
0,-97.638002,1396.497378,517.788917



💡 CIO 체크포인트:
1. TMP_Y=1(부실군)의 영업이익률(OPERATING_MARGIN)이 정상군보다 낮은가?
2. TMP_Y=1(부실군)의 부채비율(DEBT_RATIO)이 정상군보다 현저히 높은가?
👉 두 질문에 'Yes'라면, 우리가 잡은 Y값은 '진짜 리스크'를 대변하는 정답지가 맞습니다.


In [ ]:
import pandas as pd
import numpy as np

print("🧪 [Target Validation v2] ID 포맷 통일 후 Y값 정밀 검증...")

# 1. 데이터 로드 (파일 경로는 지훈님의 로컬 경로에 맞춰져 있습니다)
path_overdue = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_OVERDUE.csv"
path_moratorium = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_MORATORIUM.csv"
path_master = "Final_Integrated_Credit_Master_v5_Income_Added.csv"

# ID를 정규화(앞의 0을 제거)하여 로드하는 함수
def load_with_norm_id(path):
    df = pd.read_csv(path, dtype={'COMPANY_ID': str})
    # 앞뒤 공백 제거 및 앞자리 '0' 모두 제거 (예: 0000000072 -> 72)
    df['COMPANY_ID_NORM'] = df['COMPANY_ID'].str.strip().str.lstrip('0')
    return df

df_ov = load_with_norm_id(path_overdue)
df_mo = load_with_norm_id(path_moratorium)
df_master = load_with_norm_id(path_master)

# 2. Y값 후보군 ID 세트 생성 (정규화된 ID 기준)
ov_ids = set(df_ov['COMPANY_ID_NORM'].unique())
mo_ids = set(df_mo[df_mo['MORATORIUM_OVERDUE_AMOUNT'] > 0]['COMPANY_ID_NORM'].unique())
all_bad_ids = ov_ids | mo_ids

# 3. 마스터 파일에 Y라벨 부여 및 재무 지표 수치화
df_master['TMP_Y'] = df_master['COMPANY_ID_NORM'].apply(lambda x: 1 if x in all_bad_ids else 0)

cols_to_check = ['OPERATING_MARGIN', 'DEBT_RATIO', 'CASH_RATIO']
for col in cols_to_check:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 4. 결과 출력
print(f"\n📊 [ID 매칭 결과]")
print(f" - 리스크 데이터 총 ID 수: {len(all_bad_ids)}개")
print(f" - 마스터 파일과 최종 매칭된 부실 기업 수: {df_master['TMP_Y'].sum()}개")

if df_master['TMP_Y'].sum() > 0:
    print(f"\n📊 [최종 검증: 재무 지표 대조 (Smell Test)]")
    comparison = df_master.groupby('TMP_Y')[cols_to_check].mean()
    display(comparison)
else:
    print("❌ 여전히 매칭된 ID가 없습니다. ID 샘플을 확인해 주세요.")
    print(f"Master ID Sample: {df_master['COMPANY_ID_NORM'].head(3).tolist()}")
    print(f"Risk ID Sample: {list(all_bad_ids)[:3]}")

🧪 [Target Validation v2] ID 포맷 통일 후 Y값 정밀 검증...

📊 [ID 매칭 결과]
 - 리스크 데이터 총 ID 수: 54개
 - 마스터 파일과 최종 매칭된 부실 기업 수: 54개

📊 [최종 검증: 재무 지표 대조 (Smell Test)]


,OPERATING_MARGIN,DEBT_RATIO,CASH_RATIO
TMP_Y,,,
0,-100.233603,1413.405627,535.537116
1,-24.295641,784.781071,9.584408


In [6]:
import pandas as pd
import numpy as np

print("🧪 [Target Validation v3] 아웃라이어 제거를 위한 '중앙값' 및 '거래처 수' 검증...")

# (데이터는 이전 단계에서 로드된 df_master를 그대로 사용한다고 가정합니다)
# 1. 중앙값(Median) 기준으로 재비교 (아웃라이어 영향 배제)
cols_to_check = ['OPERATING_MARGIN', 'DEBT_RATIO', 'CASH_RATIO', 'LINKED_PARTNERS']

# 2. 결과 출력
print(f"\n📊 [최종 검증: 중앙값(Median) 대조]")
print("-" * 60)
comparison_med = df_master.groupby('TMP_Y')[cols_to_check].median()
display(comparison_med)

# 3. 데이터 분포 확인 (왜 평균이 튀었는지 확인)
print("\n🔍 [데이터 분포 체크 (Max값)]")
display(df_master.groupby('TMP_Y')[cols_to_check].max())

🧪 [Target Validation v3] 아웃라이어 제거를 위한 '중앙값' 및 '거래처 수' 검증...

📊 [최종 검증: 중앙값(Median) 대조]
------------------------------------------------------------


,OPERATING_MARGIN,DEBT_RATIO,CASH_RATIO,LINKED_PARTNERS
TMP_Y,,,,
0,3.175,126.080,26.929648,1.0
1,-0.410,207.485,3.089795,2.0



🔍 [데이터 분포 체크 (Max값)]


,OPERATING_MARGIN,DEBT_RATIO,CASH_RATIO,LINKED_PARTNERS
TMP_Y,,,,
0,76.50,909090.91,158280.000000,657.0
1,27.77,8088.57,64.794336,9.0


In [ ]:
import pandas as pd
import numpy as np

print("🎯 [Target Definition] AUS v2.0 골든 라벨링 및 마스터셋 구축 중...")

# 1. 파일 경로 설정
path_overdue = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_OVERDUE.csv"
path_moratorium = r"raw_data\PROD_FLOWSCORE\PUBLIC\COMPANY_MORATORIUM.csv"
path_master = "Final_Integrated_Credit_Master_v5_Income_Added.csv"

# ID 정규화 함수 (앞자리 0 제거)
def norm_id(df):
    df['COMPANY_ID_NORM'] = df['COMPANY_ID'].astype(str).str.strip().str.lstrip('0')
    return df

# 2. 데이터 로드 및 정규화
df_ov = norm_id(pd.read_csv(path_overdue))
df_mo = norm_id(pd.read_csv(path_moratorium))
df_master = norm_id(pd.read_csv(path_master))

# 3. 리스크 ID 추출 (54개의 정수)
ov_ids = set(df_ov['COMPANY_ID_NORM'].unique())
# Moratorium 중 실질적 리스크(연체액, 계좌정지, 워크아웃)가 있는 것만 필터링
mo_risky_ids = set(df_mo[
    (df_mo['MORATORIUM_OVERDUE_AMOUNT'] > 0) | 
    (df_mo['ACCOUNT_SUSPENSION_COUNT'] > 0) | 
    (df_mo['WORKOUT_COUNT'] > 0)
]['COMPANY_ID_NORM'].unique())

final_bad_ids = ov_ids | mo_risky_ids

# 4. 최종 Y값 부여
df_master['TARGET_Y'] = df_master['COMPANY_ID_NORM'].apply(lambda x: 1 if x in final_bad_ids else 0)

# 5. [중요] Moratorium의 상세 수치들도 피처(Feature)로 활용하기 위해 병합
# 중복 ID 방지를 위해 Max값으로 집계
df_mo_features = df_mo.groupby('COMPANY_ID_NORM').agg({
    'MORATORIUM_COUNT': 'max',
    'CARD_ACCOUNT_COUNT': 'max',
    'ACCOUNT_SUSPENSION_COUNT': 'max',
    'MORATORIUM_OVERDUE_AMOUNT': 'max'
}).reset_index()

df_final_train = pd.merge(df_master, df_mo_features, on='COMPANY_ID_NORM', how='left').fillna(0)

# 6. 데이터 품질 확인
print(f"\n✅ [구축 완료] 최종 학습 데이터 제원")
print(f" - 전체 모수: {len(df_final_train):,}개 기업")
print(f" - 부실 타겟(Y=1): {df_final_train['TARGET_Y'].sum()}개 (Golden Label)")
print(f" - 정상 타겟(Y=0): {len(df_final_train) - df_final_train['TARGET_Y'].sum():,}개")
print("-" * 60)

# 최종 파일 저장
df_final_train.to_csv("AUS_v2_Golden_Training_Set.csv", index=False)
print("💾 'AUS_v2_Golden_Training_Set.csv' 저장이 완료되었습니다.")

🎯 [Target Definition] AUS v2.0 골든 라벨링 및 마스터셋 구축 중...

✅ [구축 완료] 최종 학습 데이터 제원
 - 전체 모수: 1,514개 기업
 - 부실 타겟(Y=1): 57개 (Golden Label)
 - 정상 타겟(Y=0): 1,457개
------------------------------------------------------------
💾 'AUS_v2_Golden_Training_Set.csv' 저장이 완료되었습니다.
